In [ ]:
from pathlib import Path

import polars as pl
import numpy as np
import scipy as sp

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from utils import  get_voltage_series, find_all_data_files
from dqdvs import dqdv_histogram,  dqdv_finite_differences


from config import DATA_PATH
ZENODO_DATA_PATH = Path(DATA_PATH) / Path("HALF_CELL_OCVS_ZENODO")

In [5]:
################################ LOAD DATA  ###################################
all_filepaths = ZENODO_DATA_PATH.rglob("*.parquet")
[filepath.name for filepath in all_filepaths if "nmc111" in filepath.stem]

['sintef__sintef-nmc111-R2032-customcells-000b13__20250720__gitthold__RT.bdf.parquet',
 'sintef__sintef-nmc111-R2032-customcells-1e88d8__20250625__p-ocvhold__RT.bdf.parquet',
 'sintef__sintef-nmc111-R2032-customcells-53acb0__20250625__gitthold__RT.bdf.parquet',
 'sintef__sintef-nmc111-R2032-customcells-6115bb__20250720__p-ocv__RT.bdf.parquet',
 'sintef__sintef-nmc111-R2032-customcells-6852ff__20250720__gitthold__RT.bdf.parquet',
 'sintef__sintef-nmc111-R2032-customcells-85cad7__20250720__gitt__RT.bdf.parquet',
 'sintef__sintef-nmc111-R2032-customcells-9d3ce5__20250625__p-ocv__RT.bdf.parquet',
 'sintef__sintef-nmc111-R2032-customcells-e0dd5f__20250720__p-ocvhold__RT.bdf.parquet',
 'sintef__sintef-nmc111-R2032-customcells-e5a00e__20251015__gitthold__RT.bdf.parquet']

## NCM111 with various methods

In [11]:
df = pl.read_parquet(ZENODO_DATA_PATH / Path('sintef__sintef-nmc111-R2032-customcells-1e88d8__20250625__p-ocvhold__RT.bdf.parquet'))

df_subset = (df
             .filter(pl.col('Step Index / 1')==2)
             .filter(pl.col('Cycle Count / 1')==3)
             )

q = df_subset['Cumulative Capacity / Ah'].to_numpy()
v = df_subset['Voltage / V'].to_numpy()

# Savitzky Golay 5, ref 10.1149/1945-7111/ac38f2
SG_P = 1
SG_W_LOW = 20
SG_W_HIGH = 100
SG_W_VERYHIGH = 500
v_smooth_sg_low = sp.signal.savgol_filter(v, window_length=SG_W_LOW, polyorder=SG_P)
v_smooth_sg_high = sp.signal.savgol_filter(v, window_length=SG_W_HIGH, polyorder=SG_P)
v_smooth_sg_veryhigh = sp.signal.savgol_filter(v, window_length=SG_W_VERYHIGH, polyorder=SG_P)


# # Moving average, ref: https://doi.org/10.1016/j.jpowsour.2009.08.019
# MA_W = 50
# v_smooth_ma = sp.ndimage.uniform_filter(v, size=MA_W, mode="nearest")


# Finite differences
v_fd, dqdv_fd = dqdv_finite_differences(q, v) # FD

v_sgl_fd, dqdv_sgl_fd = dqdv_finite_differences(q, v_smooth_sg_low) # SG -> FD
v_sgh_fd, dqdv_sgh_fd = dqdv_finite_differences(q, v_smooth_sg_high) # SG -> FD
v_sgvh_fd, dqdv_sgvh_fd = dqdv_finite_differences(q, v_smooth_sg_veryhigh) # SG -> FD

dqdv_fd_sgl = sp.signal.savgol_filter(dqdv_fd, window_length=SG_W_LOW, polyorder=SG_P) # FD -> SG
dqdv_fd_sgh = sp.signal.savgol_filter(dqdv_fd, window_length=SG_W_HIGH, polyorder=SG_P) # FD -> SG
dqdv_fd_sgvh = sp.signal.savgol_filter(dqdv_fd, window_length=SG_W_VERYHIGH, polyorder=SG_P) # FD -> SG


# # Central Differences
# v_cd, dqdv_cd = compute_dqdv_central_differences(q, v) # CD
# # v_sg_cd, dqdv_sg_cd = compute_dqdv_central_differences(q, v_smooth_sg) # SG -> CD
# # v_ma_cd, dqdv_ma_cd = compute_dqdv_central_differences(q, v_smooth_ma) # MA -> CD
# dqdv_cd_sg = sp.signal.savgol_filter(dqdv_cd, window_length=SG_M, polyorder=SG_P) # CD -> SG
# dqdv_cd_sg = sp.signal.savgol_filter(dqdv_cd, window_length=SG_M, polyorder=SG_P) # CD -> SG
# # dqdv_cd_ma = sp.ndimage.uniform_filter(dqdv_cd, size=MA_W, mode="nearest") #  CD -> MA


# Histograms
BIN_SIZE_LOW = 1e-3
BIN_SIZE_HIGH = 5e-3
BIN_SIZE_VHIGH = 25e-3
v_h_l, dqdv_h_l = dqdv_histogram(q, v, BIN_SIZE_LOW)
v_h_h, dqdv_h_h = dqdv_histogram(q, v, BIN_SIZE_HIGH)
v_h_vh, dqdv_h_vh = dqdv_histogram(q, v, BIN_SIZE_VHIGH)


c:\Users\eibarc\Documents\Repositories\incremental-capacity-curves\dqdvs.py:27: RuntimeWarning:

divide by zero encountered in divide



In [12]:
colors = px.colors.qualitative.Plotly

fig = make_subplots(cols=1, rows=4, shared_xaxes=True, vertical_spacing=0.01)



fig.update_layout(
    template="ggplot2",
    margin=dict(l=10, r=10, t=10, b=10),
    width=350, 
    height=800,
    legend=dict(
        orientation="h",    # horizontal
        yanchor="bottom",   # anchor legend's bottom
        y=1.02,             # place above the plot area
        xanchor="center",
        x=0.5
    ),)



fig.add_trace(go.Scatter(x=v_fd, 
                         y=dqdv_fd, 
                         name="FD", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color="Black", width=1)), row=1, col=1)




fig.add_trace(go.Scatter(x=v_sgl_fd, 
                         y=dqdv_sgl_fd, 
                         name=f"SG {SG_W_LOW} -> FD", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[0], width=1)), row=3, col=1)

fig.add_trace(go.Scatter(x=v_sgh_fd, 
                         y=dqdv_sgh_fd, 
                         name=f"SG {SG_W_HIGH} -> FD", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[1], width=1)), row=3, col=1)

fig.add_trace(go.Scatter(x=v_sgvh_fd, 
                         y=dqdv_sgvh_fd, 
                         name=f"SG {SG_W_VERYHIGH} -> FD", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[2], width=1)), row=3, col=1)



fig.add_trace(go.Scatter(x=v_fd, 
                         y=dqdv_fd_sgl, 
                         name=f"FD -> SG {SG_W_LOW}", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[0], width=1)), row=2, col=1)

fig.add_trace(go.Scatter(x=v_fd, 
                         y=dqdv_fd_sgh, 
                         name=f"FD -> SG {SG_W_HIGH}", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[1], width=1)), row=2, col=1)

fig.add_trace(go.Scatter(x=v_fd, 
                         y=dqdv_fd_sgvh, 
                         name=f"FD -> SG {SG_W_VERYHIGH}", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[2], width=1)), row=2, col=1)


fig.add_trace(go.Scatter(x=v_h_l, 
                         y=dqdv_h_l, 
                         name=f"H {BIN_SIZE_LOW}", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[0], width=1)), row=4, col=1)

fig.add_trace(go.Scatter(x=v_h_h, 
                         y=dqdv_h_h, 
                         name=f"H {BIN_SIZE_LOW}", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[1], width=1)), row=4, col=1)

fig.add_trace(go.Scatter(x=v_h_vh, 
                         y=dqdv_h_vh, 
                         name=f"H {BIN_SIZE_LOW}", 
                         mode="lines",
                         showlegend=False,
                         line=dict(color=colors[2], width=1)), row=4, col=1)


# ANNOTATIONS
X_POS = 3.975
FONT_SIZE = 13
MAIN_FONT_WEIGHT = 150


# FD
fig.add_annotation(x=X_POS, 
                    y=0.3,
                    text="Finite<br>Differences",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=MAIN_FONT_WEIGHT), 
                    xanchor="left",
                    align="left",
                    row=1, col=1 )


# SG -> FD 
dqdv_anchor = 0.018
fig.add_annotation(x=X_POS-0.06, 
                    y=dqdv_anchor,
                    text="Smoothing then <br>Finite Differences",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=MAIN_FONT_WEIGHT), 
                    xanchor="left",
                    align="left",
                    row=3, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.85*dqdv_anchor,
                    text=f"{SG_W_LOW} pts.",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[0]), 
                    xanchor="left",
                    row=3, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.75*dqdv_anchor,
                    text=f"{SG_W_HIGH} pts.",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[1]), 
                    xanchor="left",
                    row=3, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.65*dqdv_anchor,
                    text=f"{SG_W_VERYHIGH} pts.",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[2]), 
                    xanchor="left",
                    row=3, col=1 )


# FD -> SG
dqdv_anchor = 0.043
fig.add_annotation(x=X_POS-0.06, 
                    y=dqdv_anchor,
                    text="Finite Differences<br> then Smoothing",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=MAIN_FONT_WEIGHT), 
                    xanchor="left",
                    align="left",
                    row=2, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.73*dqdv_anchor,
                    text=f"{SG_W_LOW} pts.",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[0]), 
                    xanchor="left",
                    row=2, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.57*dqdv_anchor,
                    text=f"{SG_W_HIGH} pts.",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[1]), 
                    xanchor="left",
                    row=2, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.4*dqdv_anchor,
                    text=f"{SG_W_VERYHIGH} pts.",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[2]), 
                    xanchor="left",
                    row=2, col=1 )

# H
dqdv_anchor = 0.0117
fig.add_annotation(x=X_POS-0.06, 
                    y=dqdv_anchor,
                    text="Histogram-based",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=MAIN_FONT_WEIGHT), 
                    xanchor="left",
                    row=4, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.9*dqdv_anchor,
                    text=f"{1e3*BIN_SIZE_LOW} mV",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[0]), 
                    xanchor="left",
                    row=4, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.8*dqdv_anchor,
                    text=f"{1e3*BIN_SIZE_HIGH} mV",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[1]), 
                    xanchor="left",
                    row=4, col=1 )

fig.add_annotation(x=X_POS, 
                    y=0.7*dqdv_anchor,
                    text=f"{1e3*BIN_SIZE_VHIGH} mV",
                    showarrow=False,
                    font=dict(size=FONT_SIZE, weight=int(MAIN_FONT_WEIGHT*0.5), color=colors[2]), 
                    xanchor="left",
                    row=4, col=1 )

fig.update_xaxes(showgrid=True, ticks="", tickfont=dict(size=14))
fig.update_xaxes(title=dict(text="Voltage / V", font=dict(size=16)), col=1, row=4)

fig.update_yaxes(showgrid=False,  ticks="", tickfont=dict(size=14))
fig.update_yaxes(title=dict(text="dq/dV / V<sup>-1<sup>", font=dict(size=16)))
